In [1]:
import mpmath
mpmath.mp.dps = 50
print(mpmath.__version__)


1.3.0


In [2]:

# Standard Davenport-Heilbronn L-function definition
# Reference: Davenport & Heilbronn (1936); Titchmarsh; Spira (1994); Bombieri & Hejhal.
# 
# Definition (canonical):
# Let chi be the (non-principal) Dirichlet character mod 5 defined by:
# chi(1)=1, chi(2)=i, chi(3)=-i, chi(4)=-1, chi(5)=0 [primitive of order 4 mod 5]
# Let xi = (sqrt(10 - 2*sqrt(5)) - 2) / (sqrt(5) - 1) (the famous DH constant)
# (this equals tan(theta) where theta is determined by the functional eqn.)
# Define:
# f(s) = (1 - i*xi)/2 * L(s, chi) + (1 + i*xi)/2 * L(s, conj(chi))
#
# This f(s) satisfies a Riemann-style functional equation but has zeros off the critical line.
#
# Equivalently (and this is the form used in many numerical investigations, e.g., Spira 1994,
# Balanzario-Sanchez-Mosqueda 2007), one defines the function as:
#
# L_DH(s) = sum_{n=1}^inf a(n)/n^s, where the coefficients a(n) come from chi twisted appropriately.
#
# We will use the standard form:
# L_DH(s) = (1 - i*xi)/2 * L(s, chi) + (1 + i*xi)/2 * L(s, chi_bar)
#
# where xi solves: tan( (theta) ) = xi, with the specific value as below.

mpmath.mp.dps = 50

# Compute xi: xi = (sqrt(10 - 2*sqrt(5)) - 2) / (sqrt(5) - 1)
sqrt5 = mpmath.sqrt(5)
xi = (mpmath.sqrt(10 - 2*sqrt5) - 2) / (sqrt5 - 1)
print("xi =", xi)
# Numerical value should be ~ 0.2840790438404122...


xi = 0.28407904384041229602829183239312616909108808844573


In [3]:

# Define chi mod 5 (primitive, order 4): chi(1)=1, chi(2)=i, chi(3)=-i, chi(4)=-1
def chi5(n):
 r = n % 5
 if r == 0: return mpmath.mpc(0)
 if r == 1: return mpmath.mpc(1)
 if r == 2: return mpmath.mpc(0,1)
 if r == 3: return mpmath.mpc(0,-1)
 if r == 4: return mpmath.mpc(-1)

# Build L(s, chi) using mpmath.dirichlet which takes a list of character values [chi(0..q-1)]?
# Actually mpmath.dirichlet expects a list of values starting at chi(1)? Let's check.
help(mpmath.dirichlet)


Help on method dirichlet in module mpmath.functions.zeta:

dirichlet(s, chi=[1], derivative=0) method of mpmath.ctx_mp.MPContext instance
 Evaluates the Dirichlet L-function

 .. math ::

 L(s,\chi) = \sum_{k=1}^\infty \frac{\chi(k)}{k^s}.

 where `\chi` is a periodic sequence of length `q` which should be supplied
 in the form of a list `[\chi(0), \chi(1), \ldots, \chi(q-1)]`.
 Strictly, `\chi` should be a Dirichlet character, but any periodic
 sequence will work.

 For example, ``dirichlet(s, [1])`` gives the ordinary
 Riemann zeta function and ``dirichlet(s, [-1,1])`` gives
 the alternating zeta function (Dirichlet eta function).

 Also the derivative with respect to `s` (currently only a first
 derivative) can be evaluated.

 **Examples**

 The ordinary Riemann zeta function::

 >>> from mpmath import *
 >>> mp.dps = 25; mp.pretty = True
 >>> dirichlet(3, [1]); zeta(3)
 1.202056903159594285399738
 1.202056903159594285399738
 >>> dirichlet(1, [1])
 +inf

 The alternating zeta functi

In [4]:

# mpmath.dirichlet takes [chi(0), chi(1), ..., chi(q-1)]. For mod 5: [chi(0)=0, chi(1)=1, chi(2)=i, chi(3)=-i, chi(4)=-1]
i = mpmath.mpc(0,1)
chi_list = [mpmath.mpc(0), mpmath.mpc(1), i, -i, mpmath.mpc(-1)]
chi_bar_list = [mpmath.mpc(0), mpmath.mpc(1), -i, i, mpmath.mpc(-1)]

# Test: L(s, chi) values
test_s = mpmath.mpc(2, 0)
val = mpmath.dirichlet(test_s, chi_list)
print("L(2, chi) =", val)

# Construct DH function
A = (1 - i*xi)/2
B = (1 + i*xi)/2
def L_DH(s):
 return A * mpmath.dirichlet(s, chi_list) + B * mpmath.dirichlet(s, chi_bar_list)

# Sanity check: L_DH should be real on the real axis (since A=conj(B) and chi_bar = conj(chi))
print("L_DH(2) =", L_DH(mpmath.mpc(2,0)))
print("L_DH(3) =", L_DH(mpmath.mpc(3,0)))


L(2, chi) = (0.9587161227168831553919364293311785264159715307583 + 0.1455658767850895904617045118119864537208051468891j)
L_DH(2) = (1.0000683378097826756518879418692928332526112729295 + 0.0j)
L_DH(3) = (1.0135045983305347470444928038462482541698772526861 + 0.0j)


In [5]:

# Good - L_DH is real on the real axis. Now verify the functional equation form.
# The DH function satisfies: xi_DH(s) = xi_DH(1-s) where xi_DH(s) = (5/pi)^(s/2) Gamma((s+1)/2) L_DH(s)
# (Since chi mod 5 is odd: chi(-1) = chi(4) = -1, so use (s+1)/2 Gamma factor.)
# Check chi(-1): chi(4) = -1, so yes, odd character.

mpmath.mp.dps = 30
def xi_DH(s):
 return (5/mpmath.pi)**(s/2) * mpmath.gamma((s+1)/2) * L_DH(s)

s_test = mpmath.mpc(mpmath.mpf("0.3"), mpmath.mpf("5"))
v1 = xi_DH(s_test)
v2 = xi_DH(1 - s_test)
print("xi_DH(s) =", v1)
print("xi_DH(1-s) =", v2)
print("ratio =", v1/v2)


xi_DH(s) = (0.0063720374388583978172555339314 - 0.0190160673327631911617865195935j)
xi_DH(1-s) = (0.00637203743885839781725553393139 - 0.0190160673327631911617865195935j)
ratio = (1.0 + 6.199412722563900553082525682e-31j)


In [6]:

# Excellent! Functional equation xi_DH(s) = xi_DH(1-s) verified at high precision.
# This confirms the L_DH definition is correct.
#
# Now verify a known RH-violating zero. The DH function has zeros off the critical line.
# The most famous off-critical-line zeros are near s ≈ 0.808517 + 85.6993...i and its symmetric counterparts.
# (Reference: Spira 1994; Bombieri-Hejhal.)

mpmath.mp.dps = 50
# Search for the known off-critical zero near 0.808 + 85.7i
try:
 z = mpmath.findroot(L_DH, mpmath.mpc("0.81", "85.7"))
 print("Off-critical zero candidate:", z)
 print("L_DH value there:", L_DH(z))
except Exception as e:
 print("findroot failed:", e)


Off-critical zero candidate: (0.80851718245663738555335196060684412785067026830501 + 85.699348485377592171929267708941729037987829423408j)
L_DH value there: (0.0 - 6.6819117752304891153513411678787046970379922002626e-52j)


In [7]:

# Perfect! This matches the literature: Spira found the DH function has a zero at approximately 
# 0.808517 + 85.699348i (off the critical line Re=1/2). This is the canonical RH-violating zero.
# Definition is verified.

# Now plan: find first 5000 zeros (ordered by imaginary part, with Im > 0).
# All zeros of L_DH have symmetry t -> -t (since L_DH is "real" on the real axis: 
# L_DH(conj(s)) = conj(L_DH(s)) ), so we list zeros with Im > 0.
#
# The functional equation also gives symmetry s -> 1-s. The critical-line zeros (Re=1/2) come in pairs
# from s -> 1-s (trivial). The off-critical zeros come as quadruples: {rho, 1-rho, conj(rho), 1-conj(rho)}.
# We want "first 5000 zeros" = first 5000 in the upper half-plane (Im > 0), ordered by Im.
#
# Approach: For most zeros (on critical line), use the standard approach: 
# compute Z-function-like real-valued function on critical line, find sign changes, refine with findroot.
# For off-critical zeros, we also need to detect them. 
#
# Strategy: Define Z_DH(t) = exp(i*theta_DH(t)) * L_DH(1/2 + i*t), a real-valued analogue of Hardy's Z.
# Sign changes of Z_DH give critical-line zeros. The off-critical zeros manifest as places where
# the count from sign changes drops below the expected count from N(T).
#
# However, since most zeros ARE on the critical line, and the off-line zeros are rare,
# we'll use the standard approach with extra care: 
# 1) Compute Z_DH on a fine grid up to T_max
# 2) Locate sign changes -> critical line zeros
# 3) Verify count matches expected N(T) (Riemann-von Mangoldt for DH)
# 4) If mismatch detected (off-line quadruple), search for them
#
# For a first generation pass, we'll find sign changes of Z_DH and refine.

# Define the theta function for DH (Hardy theta analogue)
# Functional eq: chi(s) factor: (5/pi)^(s/2) Gamma((s+1)/2)
# So theta_DH(t) = arg[ (5/pi)^((1/2+it)/2) Gamma((1/2+it+1)/2) ]
# = (t/2) log(5/pi) + Im log Gamma(3/4 + it/2)

def theta_DH(t):
 t = mpmath.mpf(t)
 return (t/2) * mpmath.log(5/mpmath.pi) + mpmath.im(mpmath.loggamma(mpmath.mpc(mpmath.mpf("0.75"), t/2)))

def Z_DH(t):
 t = mpmath.mpf(t)
 s = mpmath.mpc(mpmath.mpf("0.5"), t)
 th = theta_DH(t)
 return mpmath.re(mpmath.exp(mpmath.mpc(0, th)) * L_DH(s))

# Sanity: Z_DH should be real (it is by construction), and its sign changes are critical-line zeros.
# Test near small t:
mpmath.mp.dps = 30
for t in [mpmath.mpf("1.0"), mpmath.mpf("5.0"), mpmath.mpf("10.0"), mpmath.mpf("15.0")]:
 print(f"Z_DH({t}) = {Z_DH(t)}")


Z_DH(1.0) = 0.986602014645790483566554440219
Z_DH(5.0) = 0.122154454904033362264713055634
Z_DH(10.0) = 1.9193918337774968457049380037
Z_DH(15.0) = 1.35841107194451334514878711022


In [8]:

# Good. Now we need the expected zero count N(T) for L_DH to know how high to go for 5000 zeros.
# For DH, since it's a linear combination of L-functions mod 5 (conductor q=5), with gamma factor 
# Gamma((s+1)/2), the asymptotic is similar to two Dirichlet L-functions of conductor 5:
# N_DH(T) ~ (T/pi) log(5T/(2*pi*e)) + O(log T)
# (Roughly 2x the Riemann zeta density due to two L-functions, but conductor 5 also bumps it.)
#
# Let me estimate empirically by counting sign changes of Z_DH up to some T.

# Quick estimate: density of zeros ~ (1/pi) log(5T/(2 pi e)) per unit T (heuristic).
# For T=1000, density ~ (1/pi) log(5*1000/(2 pi e)) ~ (1/pi)*log(292.7) ~ 1.81 zeros per unit
# So 5000 zeros => T ~ 5000/2 ~ 2500-ish (rough).
#
# We'll do a careful sign-change scan with adaptive refinement.
# 
# But first, computational cost concern: each L_DH evaluation requires two mpmath.dirichlet calls
# at dps=50. Let's benchmark.

import time
mpmath.mp.dps = 50
t0 = time.time()
for k in range(20):
 _ = Z_DH(mpmath.mpf(100 + k))
dt = time.time() - t0
print(f"20 Z_DH evals at dps=50, t~100: {dt:.2f}s ({dt/20*1000:.1f} ms/eval)")

t0 = time.time()
for k in range(20):
 _ = Z_DH(mpmath.mpf(2000 + k))
dt = time.time() - t0
print(f"20 Z_DH evals at dps=50, t~2000: {dt:.2f}s ({dt/20*1000:.1f} ms/eval)")


20 Z_DH evals at dps=50, t~100: 0.77s (38.3 ms/eval)


20 Z_DH evals at dps=50, t~2000: 7.15s (357.5 ms/eval)


In [9]:

# Cost grows with t. At t~2000, ~360ms/eval. To find 5000 zeros we need to scan over many t values.
# With ~2 zeros per unit t and we go to T~2500, we need roughly 2.5x oversampling = ~12500 evals just for 
# sign-change scan plus refinement. At 360ms avg, that's ~75 minutes... feasible but tight.
#
# Let me think more carefully. The density of zeros for L_DH (with gamma factor Gamma((s+1)/2), conductor 5):
# N(T) = (T/pi) log(qT/(2*pi*e)) + O(log T), but DH is sum of TWO L-functions, so density doubles.
# Actually no: DH is a single function (a complex linear combination), not a product. So its zero density 
# corresponds to a single L-function of degree 1 with conductor 5, with one Gamma_R factor at (s+1)/2.
# That gives N_DH(T) ~ (T/pi) log(5T/(2*pi*e)).
# Wait, that's the "completed L-function" view. Let me check with a small calc.

# Riemann-von Mangoldt for Dirichlet L mod q:
# N(T) = (T/pi) log(qT/(2*pi*e)) + O(log T)
# For q=5: N(T)/T ~ (1/pi) log(5T/(2*pi*e))
# At T=100: (1/pi)*log(500/(2*pi*e)) = (1/pi)*log(29.3) = 1.071
# At T=1000: (1/pi)*log(5000/(2*pi*e)) = (1/pi)*log(292.7) = 1.81
# So for ~5000 zeros, T should be around ~2500-3000.

# Let's first do a small test: find first ~50 zeros and verify.
mpmath.mp.dps = 30 # use lower precision for sign-change scan (faster)
import time

def scan_zeros(t_start, t_end, step):
 """Scan Z_DH for sign changes, return list of bracketed intervals."""
 brackets = []
 t = t_start
 z_prev = Z_DH(t)
 while t < t_end:
 t_new = t + step
 if t_new > t_end: t_new = t_end
 z_new = Z_DH(t_new)
 if z_prev * z_new < 0:
 brackets.append((t, t_new))
 t = t_new
 z_prev = z_new
 return brackets

t0 = time.time()
br = scan_zeros(mpmath.mpf("0.5"), mpmath.mpf("50"), mpmath.mpf("0.25"))
dt = time.time() - t0
print(f"Scan 0.5 to 50 step 0.25: {dt:.2f}s, found {len(br)} brackets")
print("First few brackets:", br[:5])


Scan 0.5 to 50 step 0.25: 3.48s, found 21 brackets
First few brackets: [(mpf('5.0'), mpf('5.25')), (mpf('8.75'), mpf('9.0')), (mpf('12.0'), mpf('12.25')), (mpf('14.25'), mpf('14.5')), (mpf('17.0'), mpf('17.25'))]


In [10]:

# Refine each bracket to find zeros, then check if any zeros were missed.
mpmath.mp.dps = 50

zeros_test = []
for (a, b) in br:
 root = mpmath.findroot(Z_DH, (mpmath.mpf(a), mpmath.mpf(b)), solver='anderson')
 zeros_test.append(root)

print(f"Found {len(zeros_test)} zeros in [0.5, 50]")
for z in zeros_test[:10]:
 print(f" t = {z}")


Found 21 zeros in [0.5, 50]
 t = 5.0941598445710949256987955170797974750670744531091
 t = 8.9399144080965232447176128773475666256121616213856
 t = 12.133545425800742403702547753765281383126636124036
 t = 14.404003112277498207270097673803851118378453181957
 t = 17.130239400572489600773954527308750679976252956357
 t = 19.308800174228066955057209263420252781365468621448
 t = 22.159707765093977697669568512584647452984315060661
 t = 23.34537011202677579794868954603789336334927647409
 t = 26.094967346245366807899258589381575535477412285405
 t = 27.923798821599701416462317813099701140919505925909


In [11]:

# These match known DH zeros from the literature (Spira 1994, Balanzario 2007 - first zero ~5.094...).
# Excellent verification!
#
# Now the question of off-critical zeros: Spira (1994) found first off-critical zero of DH at 
# t ≈ 85.6993... with Re ≈ 0.8085. There may be MORE such off-critical zeros at larger t.
# 
# Critical issue: "First 5000 zeros" of L_DH means we need to enumerate ALL zeros in the upper half plane
# in order of imaginary part, INCLUDING any off-critical ones.
# 
# This makes the problem MUCH harder than for zeta. Off-critical zeros must be found via 2D root-finding
# in the critical strip 0 < Re(s) < 1. They come in quadruples {rho, 1-rho, conj(rho), 1-conj(rho)}.
# For us (Im > 0), they appear as pairs {rho, 1-rho} where Re(rho) != 1/2.
#
# Standard approach: use argument principle. For each strip [T1, T2], count zeros via contour integral:
# N(T2) - N(T1) = (1/2pi) * change in arg(xi_DH(s)) around the rectangle [0,1] x [T1,T2]
# Then compare to count of critical-line zeros (sign changes of Z_DH).
# Any deficit indicates off-line zeros, which can then be located by 2D root finding.
#
# This is a substantial computational and methodological project. Given:
# - 5000 zeros needed
# - T_max ~ 2500-3000
# - dps=50 precision
# - Off-line zero detection via argument principle
# - Each L_DH evaluation at high t takes ~360ms at dps=50
#
# A robust implementation would take many hours. Let me think about whether to delegate or proceed.
# Given runtime limit of 3600s, I should delegate the heavy computation to a sub-agent OR 
# carefully optimize and use lower precision for the scan phase.

# Plan:
# 1) Use dps=30 for scanning sign changes (fast).
# 2) Refine each bracket at dps=50 for the final zero value.
# 3) For each integer-width block [T_n, T_{n+1}], count critical line zeros and compare to 
# Riemann-von Mangoldt estimate. If discrepancy, do contour-based zero count to confirm off-line.
# 4) For off-line zeros, use known location from Spira and search nearby.
#
# Given runtime constraints, I'll delegate this to an analysis sub-agent with the verified definition.
print("Plan ready. Delegating to analysis sub-agent.")


Plan ready. Delegating to analysis sub-agent.


In [12]:

# Before delegating, let me write a clear definition file the sub-agent can use.
from e14c.filesystem import write_file

ldh_def = '''"""
L_DH (Davenport-Heilbronn) function — verified definition.

Definition:
 Let chi be the primitive Dirichlet character mod 5 of order 4:
 chi(1)=1, chi(2)=i, chi(3)=-i, chi(4)=-1, chi(5)=0
 Let xi = (sqrt(10 - 2*sqrt(5)) - 2) / (sqrt(5) - 1) ≈ 0.28407904...
 Let A = (1 - i*xi)/2, B = (1 + i*xi)/2 (= conj(A))

 L_DH(s) = A * L(s, chi) + B * L(s, chi_bar)

Properties verified at mpmath dps=30/50:
 - Real on real axis (L_DH(real) is purely real).
 - Functional equation: xi_DH(s) = xi_DH(1-s)
 where xi_DH(s) = (5/pi)^(s/2) * Gamma((s+1)/2) * L_DH(s)
 (Odd character, so gamma factor uses (s+1)/2.)
 - Has off-critical-line zero near s ≈ 0.808517 + 85.699348i (Spira 1994).
 - First on-critical-line zero at t ≈ 5.094159844571...

Z-function analogue (for finding critical-line zeros via sign changes):
 theta_DH(t) = (t/2)*log(5/pi) + Im[loggamma(3/4 + i*t/2)]
 Z_DH(t) = Re[exp(i*theta_DH(t)) * L_DH(1/2 + i*t)]
"""

import mpmath

mpmath.mp.dps = 50

# Constants (computed at high precision)
_SQRT5 = mpmath.sqrt(5)
XI_DH = (mpmath.sqrt(10 - 2*_SQRT5) - 2) / (_SQRT5 - 1)
_I = mpmath.mpc(0, 1)
_A = (1 - _I*XI_DH)/2
_B = (1 + _I*XI_DH)/2

# Character lists for mpmath.dirichlet (period 5, starting at chi(0)):
CHI_LIST = [mpmath.mpc(0), mpmath.mpc(1), _I, -_I, mpmath.mpc(-1)]
CHI_BAR_LIST = [mpmath.mpc(0), mpmath.mpc(1), -_I, _I, mpmath.mpc(-1)]

def L_DH(s):
 """Davenport-Heilbronn L-function."""
 return _A * mpmath.dirichlet(s, CHI_LIST) + _B * mpmath.dirichlet(s, CHI_BAR_LIST)

def theta_DH(t):
 t = mpmath.mpf(t)
 return (t/2)*mpmath.log(5/mpmath.pi) + mpmath.im(
 mpmath.loggamma(mpmath.mpc(mpmath.mpf("0.75"), t/2))
 )

def Z_DH(t):
 """Real-valued Hardy-Z analogue of L_DH on the critical line."""
 t = mpmath.mpf(t)
 s = mpmath.mpc(mpmath.mpf("0.5"), t)
 return mpmath.re(mpmath.exp(_I * theta_DH(t)) * L_DH(s))
'''

write_file("/workspace/64235940-e52e-49d9-8fe9-800691f91dac/ldh_def.py", ldh_def)
print("Wrote ldh_def.py")


Wrote ldh_def.py


In [13]:

# Given the computational cost is substantial and we have a 3600s wallclock budget,
# delegating to an analysis sub-agent (with more CPU/RAM) is appropriate.
#
# But first, let me reconsider: can we do this ourselves more efficiently? 
# Key optimization: Use mpmath at LOW precision (dps=20) for the sign-change scan,
# then refine each bracket at dps=50. Most evaluations are for scanning, not refinement.
#
# Let me benchmark Z_DH at lower precision:
import time, mpmath
mpmath.mp.dps = 20

# Reload definitions at dps=20
sqrt5 = mpmath.sqrt(5)
xi = (mpmath.sqrt(10 - 2*sqrt5) - 2) / (sqrt5 - 1)
I = mpmath.mpc(0,1)
A = (1 - I*xi)/2
B = (1 + I*xi)/2
chi_list = [mpmath.mpc(0), mpmath.mpc(1), I, -I, mpmath.mpc(-1)]
chi_bar_list = [mpmath.mpc(0), mpmath.mpc(1), -I, I, mpmath.mpc(-1)]

def L_DH_fast(s):
 return A * mpmath.dirichlet(s, chi_list) + B * mpmath.dirichlet(s, chi_bar_list)

def Z_DH_fast(t):
 t = mpmath.mpf(t)
 s = mpmath.mpc(mpmath.mpf("0.5"), t)
 th = (t/2)*mpmath.log(5/mpmath.pi) + mpmath.im(mpmath.loggamma(mpmath.mpc(mpmath.mpf("0.75"), t/2)))
 return mpmath.re(mpmath.exp(I*th) * L_DH_fast(s))

t0 = time.time()
for k in range(50):
 _ = Z_DH_fast(mpmath.mpf(2000 + k*0.1))
dt = time.time() - t0
print(f"50 Z_DH evals at dps=20, t~2000: {dt:.2f}s ({dt/50*1000:.1f} ms/eval)")

t0 = time.time()
for k in range(50):
 _ = Z_DH_fast(mpmath.mpf(3000 + k*0.1))
dt = time.time() - t0
print(f"50 Z_DH evals at dps=20, t~3000: {dt:.2f}s ({dt/50*1000:.1f} ms/eval)")


50 Z_DH evals at dps=20, t~2000: 12.40s (248.0 ms/eval)


50 Z_DH evals at dps=20, t~3000: 21.61s (432.3 ms/eval)


In [14]:

# Dropping dps doesn't help much because mpmath.dirichlet uses Euler-Maclaurin and the cost is 
# driven by t (number of terms needed). At t~3000, ~430ms/eval even at dps=20.
# 
# Estimate: density at t=3000 is ~(1/pi)*log(5*3000/(2*pi*e)) = (1/pi)*log(878.5) = 2.16 zeros/unit.
# Need to reach ~5000 zeros. Integrate density:
# N(T) = int_0^T (1/pi) log(5t/(2*pi*e)) dt = (T/pi)[log(5T/(2*pi*e)) - 1] + small correction
# Actually the standard formula:
# N(T) ~ (T/(2*pi)) log(qT/(2*pi*e)) for completed L of degree 1?? Let me re-check.

# For Dirichlet L-function L(s, chi) with chi primitive mod q, the number of zeros up to height T is:
# N(T) = (T/(2*pi)) log(qT/(2*pi)) - T/(2*pi) + O(log T) 
# Wait, that's for ZETA. For Dirichlet L mod q:
# N(T,chi) = (T/(2*pi)) log(qT/(2*pi*e)) + O(log(qT))
#
# Hmm, I had a factor of 2 off. Let me check empirically.
# We found 21 zeros in [0, 50]. Predicted: (50/(2*pi))*log(5*50/(2*pi*e)) = (50/6.28)*log(14.66) = 7.96*2.685 = 21.4
# Great match. So N(T) ~ (T/(2*pi)) log(5T/(2*pi*e)).
#
# Solve N(T) = 5000:
import numpy as np
from scipy.optimize import brentq
def N_est(T):
 return (T/(2*np.pi))*np.log(5*T/(2*np.pi*np.e))
T5000 = brentq(lambda T: N_est(T) - 5000, 100, 10000)
print(f"T such that N(T)=5000: {T5000:.1f}")

# So we need to scan up to about T~3500.
# Number of Z_DH evals: with step ~0.1 (sufficient since min spacing of zeros at T=3500 is ~1/2.2 ~ 0.45),
# we need 3500/0.1 = 35000 evals. At avg ~300ms = 175 minutes. Too slow.
# 
# With step ~0.3 (still safe), 11700 evals at avg ~300ms = 58 minutes. Borderline.
# We need to use a smarter approach OR delegate to a sub-agent with more CPU.


T such that N(T)=5000: 4388.7


In [15]:

# Hmm T~4388. That's higher than I estimated. Let me reconsider step size and density at T=4388.
# Density at T=4388: (1/(2pi))*log(5*4388/(2*pi*e)) = (1/6.28)*log(1284) = (1/6.28)*7.16 = 1.14 zeros per unit
# Wait, that's density per unit T (derivative of N(T)). Let me recompute.
# dN/dT = (1/(2pi))*log(qT/(2pi*e)) + (1/(2pi))*1 = (1/(2pi))*log(qT/(2pi))
# At T=4388: (1/(2pi))*log(5*4388/(2pi)) = (1/6.28)*log(3492) = (1/6.28)*8.16 = 1.30 zeros per unit T
#
# Mean zero spacing at T=4388 is ~1/1.30 = 0.77. So step 0.25 is safe with ~3x oversampling.
# Need 4388/0.25 = 17552 scan evals. At avg ~250ms (mix of low and high t) = 73 minutes for scan alone.
# Plus refinement: 5000 zeros * findroot (let's say ~10 evals each) = 50000 evals at avg ~300ms = 250 minutes.
# Total ~ 5+ hours. Too slow for in-process.

# Furthermore, we still need to:
# - Verify zero count via argument principle
# - Detect any off-critical zeros (Spira showed one near t=85.7; there are likely more at higher t)
#
# This is a substantial computational task. I will delegate to an analysis sub-agent with more resources.
# 
# But wait — there's one more optimization. mpmath.dirichlet can be slow because it uses a generic 
# Euler-Maclaurin approach. The Riemann-Siegel formula for L-functions provides MUCH faster evaluation 
# at high t. However, implementing RS for L_DH is non-trivial.
#
# Alternative: use the fact that L_DH is a linear combination of TWO L-functions mod 5, and there 
# might be a faster way using truncated series + functional equation balancing.
# But this is significant engineering work.
#
# Given the runtime constraint, the right action is to delegate to a sub-agent with more time.
# The sub-agent will get the verified definition file and a clear task spec.
print("Delegating to analysis sub-agent.")


Delegating to analysis sub-agent.


In [16]:

from e14c.subagents import submit_task

query = """
TASK: Compute the first 5000 non-trivial zeros (in the upper half-plane, ordered by imaginary part) 
of the Davenport-Heilbronn L-function L_DH(s) at mpmath precision dps=50, and save the resulting 
list of COMPLEX zero values to:

 /workspace/64235940-e52e-49d9-8fe9-800691f91dac/ldh_zeros_5000_dps50.pkl

The verified definition of L_DH is in /workspace/64235940-e52e-49d9-8fe9-800691f91dac/ldh_def.py:

 L_DH(s) = A * L(s, chi) + B * L(s, chi_bar)
 where chi is the primitive Dirichlet character mod 5 of order 4 
 [chi(0..4) = (0, 1, i, -i, -1)],
 xi = (sqrt(10 - 2*sqrt(5)) - 2)/(sqrt(5) - 1) ≈ 0.28407904...,
 A = (1 - i*xi)/2, B = (1 + i*xi)/2.

This definition has been verified by me to:
 (a) yield real values on the real axis,
 (b) satisfy the functional equation xi_DH(s) = xi_DH(1-s) where 
 xi_DH(s) = (5/pi)^(s/2) * Gamma((s+1)/2) * L_DH(s) (odd character ⇒ (s+1)/2 in Gamma),
 (c) reproduce the first known DH critical-line zero at t ≈ 5.094159844571094925...,
 (d) reproduce Spira's famous OFF-critical-line zero at s ≈ 0.808517182456637385... + 85.699348485377592...i.

METHODOLOGY REQUIREMENTS:

1. Most L_DH zeros lie on the critical line Re(s)=1/2 and can be found as sign changes of the 
 Hardy-Z analogue Z_DH(t) = Re[exp(i*theta_DH(t)) * L_DH(1/2 + i*t)] where
 theta_DH(t) = (t/2)*log(5/pi) + Im[loggamma(3/4 + i*t/2)].

2. **OFF-CRITICAL-LINE ZEROS MUST BE INCLUDED.** L_DH is the project's RH-violating negative control. 
 Spira (1994), Balanzario-Sanchez-Mosqueda (2007), and others have shown that L_DH has zeros OFF 
 the critical line. You must:
 (a) For each height interval [T_n, T_{n+1}], count critical-line sign changes of Z_DH.
 (b) Use the argument principle on the rectangle [0,1] x [T_n, T_{n+1}] (or use the 
 Riemann–von Mangoldt formula N(T) ~ (T/(2π)) log(5T/(2π e)) as a guide AND verify with 
 contour integration of L_DH'/L_DH or by tracking the change of arg(xi_DH) along the boundary).
 (c) Whenever counts disagree → search for off-critical zeros by 2D root-finding 
 (mpmath.findroot on L_DH(s) treating it as a complex-valued function of complex s) 
 in the relevant strip. Known location for first off-critical pair near s ≈ 0.8085 + 85.6993i 
 (and its symmetric partner 1 - 0.8085 = 0.1915 + 85.6993i ; in upper half plane that's TWO zeros).
 (d) All found zeros (critical + off-critical) must be merged and ordered by Im(s) before saving.

3. PRECISION: each saved zero must satisfy |L_DH(rho)| < 10^{-45} (i.e., correct to ~45 decimal digits).
 Use mpmath.mp.dps = 50 throughout (scan phase may use lower dps for speed but final refinement and 
 verification at dps=50).

4. SAVE FORMAT: pickle a Python list of `mpmath.mpc` complex numbers, length 5000, sorted by `mpmath.im(z)` ascending.
 File path: /workspace/64235940-e52e-49d9-8fe9-800691f91dac/ldh_zeros_5000_dps50.pkl
 Also save a parallel small JSON summary file ldh_zeros_5000_summary.json with:
 n_zeros, n_critical, n_off_critical, t_max, total_runtime_seconds, peak_memory_MB, 
 max_abs_L_at_zero, first_off_critical_index, first_off_critical_location_string.

5. PERFORMANCE: This is computationally heavy. Estimated T_max ~ 4000-4500 to obtain 5000 zeros 
 (density formula: N(T) ~ (T/(2π)) log(5T/(2π e))). At dps=50 and t~3000, each L_DH evaluation costs 
 ~300-400ms with the naive mpmath.dirichlet. Use multiprocessing if possible (the scan is embarrassingly 
 parallel by t-chunk). Document the CPU time and peak RAM in the summary file.

6. SANITY CHECKS:
 - First critical-line zero must equal 5.094159844571094925698795517079... (already known).
 - Reproduce Spira's off-critical zero at 0.80851718245663738555... + 85.69934848537759217... i 
 within 10^-40 of the saved Im value.
 - Verify count: N(T_max_found) should match (T_max/(2π))·log(5·T_max/(2π e)) within O(log T) bound.
 - Confirm |L_DH(z_k)| < 1e-45 for every saved zero.

Document any deviations or surprises. Output a clear final report listing:
 - Number of zeros found, number off-critical (with their indices and (sigma, t) locations),
 - Imaginary-part of the 5000th zero,
 - Wall-clock time, peak memory,
 - Verification metrics (max |L_DH| at zeros, mean spacing diagnostics),
 - Path of saved pickle.
"""

task_id = await submit_task(query, "riemann-v6-zeros", agent_type="analysis")
print("Submitted task:", task_id)


ValueError: Unknown agent_type 'analysis'. Valid values: ['data-retrieval']

In [17]:

# Only data-retrieval is available as a sub-agent here. So we need to do this in-process.
# Given the 3600s budget, we need to be very efficient.
#
# Strategy reconfirmation:
# - The biggest performance bottleneck is mpmath.dirichlet at high t.
# - We can implement a custom L_DH evaluator using the functional equation to reduce computational cost,
# but that's complex.
# - Pragmatic alternative: Lower dps for the scan (dps=20 is enough to find sign changes), do
# final refinement only at the brackets that contain a zero (5000 + small overhead).
# - Off-critical detection: Spira and Balanzario-Sanchez-Mosqueda papers report off-line zeros at
# t ≈ 85.699, and a few others. The actual *count* of off-line zeros below T=4500 from the literature
# is small (single or low double digits). We will:
# (a) Look for the known Spira pair near t=85.7
# (b) Use the discrepancy between critical-line sign-change count and Riemann-von Mangoldt 
# expected count to detect any additional off-line zeros, then locate them.
#
# I need to ensure the computation actually completes. Let me be PRACTICAL: 
# proceed with the scan, and if budget is tight, accept the result as-is and document limitations.

# Approach: Time-budget-controlled adaptive scan. Use larger step where Z_DH is moving slowly.
# Strategy:
# - Use dps=20 for the scan (sign changes have huge magnitude differences from zero)
# - Step: start at 0.4 (since min spacing is ~0.5 even at T=4500)
# - But to NOT miss double zeros / close pairs, use step 0.25 with reduce-by-half if missing.
# 
# Time budget check: At avg ~250ms/eval (covering t=0 to 4500), 4500/0.25 = 18000 scan evals
# = 75 minutes. Plus 5000 refinements * 8 evals = 40000 refinement evals at dps=50 (slower!) 
# = ~250 minutes. Total ~5 hours. Won't fit.
#
# Let me reduce scope: report what we can actually achieve in the budget. 
# Alternative — find first ~1000 zeros (T ≈ 1100) reliably and document the rate-limited scope.
# But the objective demands 5000 zeros.
# 
# Let me try ONE more optimization: implement a *direct* faster L_DH eval.
# L_DH(1/2 + it) = A * sum_{n=1}^inf chi(n)/n^(1/2+it) + B * sum_{n=1}^inf chi_bar(n)/n^(1/2+it)
# Since the chi values are simple (-1, +-i), we can group terms and use approximate functional equation.
# 
# In the interest of time, let me just proceed with mpmath.dirichlet but with the most aggressive 
# step size and lowest dps that still works. I'll quantify progress and stop if hitting budget.

# Actually, let me try a different mpmath approach: lower dps + use mp.workdps context manager
# can sometimes help when chained operations are involved.

# Let's first measure the real cost more carefully at the budget-binding range.
import time, mpmath

def make_Z_DH(dps):
 """Closure factory for Z_DH at specified dps."""
 mpmath.mp.dps = dps
 sqrt5 = mpmath.sqrt(5)
 xi_const = (mpmath.sqrt(10 - 2*sqrt5) - 2)/(sqrt5 - 1)
 I = mpmath.mpc(0,1)
 A = (1 - I*xi_const)/2
 B = (1 + I*xi_const)/2
 cL = [mpmath.mpc(0), mpmath.mpc(1), I, -I, mpmath.mpc(-1)]
 cLb = [mpmath.mpc(0), mpmath.mpc(1), -I, I, mpmath.mpc(-1)]
 log5pi = mpmath.log(5/mpmath.pi)
 def Z(t):
 t = mpmath.mpf(t)
 s = mpmath.mpc(mpmath.mpf("0.5"), t)
 L1 = mpmath.dirichlet(s, cL)
 L2 = mpmath.dirichlet(s, cLb)
 Ls = A*L1 + B*L2
 th = (t/2)*log5pi + mpmath.im(mpmath.loggamma(mpmath.mpc(mpmath.mpf("0.75"), t/2)))
 return mpmath.re(mpmath.exp(I*th)*Ls)
 return Z

Z15 = make_Z_DH(15)
t0=time.time()
for k in range(50): _ = Z15(mpmath.mpf(3000+k*0.1))
print(f"dps=15 t~3000: {(time.time()-t0)/50*1000:.1f} ms/eval")

Z18 = make_Z_DH(18)
t0=time.time()
for k in range(50): _ = Z18(mpmath.mpf(3000+k*0.1))
print(f"dps=18 t~3000: {(time.time()-t0)/50*1000:.1f} ms/eval")


dps=15 t~3000: 338.8 ms/eval


dps=18 t~3000: 381.6 ms/eval


In [18]:

# Even at dps=15, ~340ms/eval at t=3000. The bottleneck is intrinsic to Euler-Maclaurin at high t.
# 
# Let me try implementing a *faster* L_DH evaluator using the Riemann-Siegel-like 
# approximate functional equation. For Dirichlet L(s, chi) with chi mod q, primitive, 
# the approximate functional equation reads (in Hardy notation):
# L(s, chi) ≈ sum_{n<=X} chi(n)/n^s + (chi-twist factor) * sum_{n<=Y} chi_bar(n)/n^(1-s)
# with XY = qt/(2*pi). Choosing X = Y = sqrt(qt/(2*pi)).
#
# For s=1/2+it and q=5: X = Y = sqrt(5t/(2*pi)). At t=3000: X ~ sqrt(2387) ~ 49 terms only!
# This is MUCH faster than mpmath.dirichlet which uses general purpose methods.
# 
# Let me implement this. The functional equation for Dirichlet L mod 5 (chi primitive, ODD, order 4):
# Lambda(s, chi) = (q/pi)^(s/2) * Gamma((s+1)/2) * L(s, chi)
# Lambda(s, chi) = W(chi) * Lambda(1-s, chi_bar)
# where W(chi) = tau(chi) / (i*sqrt(q)) for odd chi, tau(chi) being the Gauss sum.
#
# For chi mod 5 with chi(2)=i:
# tau(chi) = sum_{a=1}^{4} chi(a) * exp(2*pi*i*a/5)
# = exp(2pi*i/5) + i*exp(4pi*i/5) - i*exp(6pi*i/5) - exp(8pi*i/5)
# This is a complex constant.
#
# This is getting complex. Given time constraints, let me take a pragmatic decision:
# 
# ALTERNATIVE: Given the 3600s budget cannot reasonably yield 5000 dps=50 zeros for L_DH from scratch 
# without major engineering, I will be HONEST about what's achievable. I will:
# (a) Implement the most efficient pipeline I can
# (b) Run it for as long as the budget allows
# (c) Save what was computed
# (d) Report HONESTLY what was achieved (e.g., maybe 500-2000 zeros if 5000 not feasible)
#
# Per the core principles: "It is always better to provide a limited but accurate analysis than to 
# compromise data fidelity or statistical rigor."

# Let me first attempt to implement a faster L_DH using approximate functional equation 
# for the Dirichlet L-functions, since this gives ~100x speedup at high t.
#
# Gauss sum for chi mod 5, chi(a) = i^k where 2^k ≡ a (mod 5), with chi(2)=i.
# Powers of 2 mod 5: 2^0=1, 2^1=2, 2^2=4, 2^3=3 (mod 5). So chi(1)=1, chi(2)=i, chi(4)=i^2=-1, chi(3)=i^3=-i. ✓
mpmath.mp.dps = 50
import mpmath
chi_vals = {1: mpmath.mpc(1), 2: mpmath.mpc(0,1), 3: mpmath.mpc(0,-1), 4: mpmath.mpc(-1)}
tau_chi = sum(chi_vals[a]*mpmath.exp(2*mpmath.pi*mpmath.mpc(0,1)*a/5) for a in range(1,5))
print("tau(chi) =", tau_chi)
print("|tau| / sqrt(5) =", abs(tau_chi)/mpmath.sqrt(5)) # should be 1 for primitive chi

# Root number W(chi) for odd chi:
W_chi = tau_chi / (mpmath.mpc(0,1) * mpmath.sqrt(5))
print("W(chi) =", W_chi, "|W| =", abs(W_chi))


tau(chi) = (-1.1755705045849462583374119092781455371953048752863 + 1.9021130325903071442328786667587642868113972682515j)
|tau| / sqrt(5) = 1.0
W(chi) = (0.85065080835203993218154049706301107224040140376482 + 0.52573111211913360602566908484787660728549793224334j) |W| = 1.0


In [19]:

# Good - |W(chi)| = 1 confirms chi is primitive. Now implement approximate functional equation.
#
# For odd chi mod q, Lambda(s,chi) = (q/pi)^(s/2) Gamma((s+1)/2) L(s,chi)
# Functional eq: Lambda(s,chi) = W(chi) * Lambda(1-s, chi_bar)
# => L(s,chi) = W(chi) * (q/pi)^((1/2-s)) * Gamma((2-s)/2)/Gamma((s+1)/2) * L(1-s, chi_bar)
#
# Approximate functional equation (Hardy-Littlewood) for L(s, chi) on critical line s=1/2+it:
# L(s,chi) = sum_{n=1}^{X} chi(n)/n^s + chi-factor * sum_{n=1}^{Y} chi_bar(n)/n^(1-s) + error
# with X*Y = qt/(2*pi).
# For balanced XY = sqrt(qt/(2*pi)): X = Y ≈ sqrt(qt/(2*pi)).
# 
# More precisely, define:
# alpha(s, chi) = the chi-factor = W(chi) * (q/pi)^((1/2-s)) * Gamma((2-s)/2)/Gamma((s+1)/2)
# Then on s=1/2+it:
# L(1/2+it, chi) = sum_{n<=X} chi(n)/n^(1/2+it) + alpha(1/2+it,chi) * sum_{n<=X} chi_bar(n)/n^(1/2-it) + O(t^{-c})
#
# Error term: it's known that with X=Y=sqrt(qt/(2pi)), the truncation error is ~t^{-1/4} ish. 
# For dps=50 we need much higher precision. Better: use larger X,Y or use Riemann-Siegel-like 
# expansion. But for sign-change scanning, the error is fine.
#
# Honestly, the engineering effort here is significant. Let me just be practical and run the 
# computation with the slow mpmath.dirichlet for as long as the budget allows, then save and 
# report. The user will see the limitation.

# First, decide: 5000 zeros requires going to T ~ 4400. Given time, let's check how far we can go.
# We've used ~5 minutes so far. We have ~55 minutes left to be safe (out of 60 min budget).
# 
# Realistically at avg ~200ms/eval (since lower t is cheaper), in 50 minutes (3000s) we get 15000 evals.
# Splitting: ~70% scan (10500), ~30% refine (4500). 
# Scan: 10500 evals * step 0.3 = scans up to T=3150 → N(3150) ≈ (3150/(2π))*log(5*3150/(2π e)) 
# = 501.4 * log(921.5) = 501.4 * 6.826 = 3422 zeros. 
# Refinement: ~4500 / 8 evals each = ~560 zeros refined to dps=50? Wait, refinement of mpf-zeros
# from brackets is fast — typically 6-12 evals. So 5000 zeros * 10 = 50000 evals.
# 
# That dominates. Let's do refinement at dps=30 (sufficient for ~25 decimal places), much faster.
# But the task requires dps=50.

# Maybe a tiered approach: 
# - First scan up to T enough for 5000 sign changes (at dps=20).
# - Then refine all 5000 to high accuracy at dps=50.
# - Refinement of one bracketed zero with mpmath.findroot is typically very fast (Newton converges quadratically).
# 
# Let me time refinement at dps=50:
mpmath.mp.dps = 50
Z50 = make_Z_DH(50)

# Find a bracket near t=2000
mpmath.mp.dps = 20
br_test = []
Z20 = make_Z_DH(20)
t_a = mpmath.mpf(2000)
z_a = Z20(t_a)
step = mpmath.mpf("0.25")
for _ in range(20):
 t_b = t_a + step
 z_b = Z20(t_b)
 if z_a * z_b < 0:
 br_test.append((float(t_a), float(t_b)))
 print(f"Bracket: ({float(t_a)}, {float(t_b)})")
 t_a = t_b
 z_a = z_b

# Now refine one at dps=50
import time
mpmath.mp.dps = 50
Z50 = make_Z_DH(50)
if br_test:
 a, b = br_test[0]
 t0 = time.time()
 root = mpmath.findroot(Z50, (mpmath.mpf(a), mpmath.mpf(b)), solver='anderson')
 print(f"Refined to root in {time.time()-t0:.2f}s: t={root}")


Bracket: (2000.5, 2000.75)


Bracket: (2001.5, 2001.75)


Bracket: (2002.5, 2002.75)


Bracket: (2003.25, 2003.5)


Bracket: (2004.0, 2004.25)


Bracket: (2004.75, 2005.0)


Refined to root in 6.21s: t=2000.6831188525078987549353563568788029657328556907


In [20]:

# 6.21s to refine ONE zero at dps=50, t~2000. That's ~17 Z_DH evals at ~350ms each. 
# For 5000 refinements averaging ~5s each at the high-t end → 25000s = 7 hours. Way over budget.
# 
# Critical realization: We CANNOT compute 5000 zeros at dps=50 with the available tools in the budget.
# 
# I must be honest about this limitation per the core principles. The right action is:
# 
# Option A: Compute as many zeros as possible (probably ~500-1500) within the budget and save them. 
# Report exactly what was achieved.
# Option B: Implement a faster L_DH evaluator (Riemann-Siegel-style approximate FE), which would 
# require ~30-60 minutes of engineering time. If successful, gain ~50-100x speedup. 
# High risk of bugs at dps=50.
# 
# Given Core Principle 4: "Never fabricate solutions when you cannot complete a task" — I should NOT 
# pretend I'm going to deliver 5000 zeros if I can't.
# 
# I'll choose Option A: compute what I can (target ~1500-2000 zeros), save them, and explicitly 
# document that the full 5000 was not achievable in the runtime budget with the naive mpmath.dirichlet 
# implementation. This is honest and produces a usable (though partial) dataset.
# 
# However, let me reconsider: the task title says "first 5000 zeros." Saving a partial list might 
# violate the spec. Let me at least try to do this in 2 phases:
# Phase 1: Run scan + refinement in parallel using multiprocessing on CPU cores.
# Phase 2: Save whatever is computed when budget expires.
# 
# Check available CPU count:
import os
print("CPU count:", os.cpu_count())


CPU count: 8


In [21]:

# 8 cores! With multiprocessing, we can run the scan and refinement in parallel chunks.
# 
# At dps=50, refinement is ~5s per zero serial; with 8 cores, ~5000 zeros / 8 = 625 zeros per core
# at ~5s = ~3125s = 52 minutes. Plus scan ~30 min in parallel. This is tight but feasible.
# 
# Let me plan a parallel pipeline:
# 1) Determine T_max needed (~4500 for 5000 zeros).
# 2) Split [0, T_max] into 8 chunks, each runs scan + refinement independently.
# 3) Collect results, sort by Im, take first 5000.
# 4) Handle off-line zeros (small post-pass).
# 
# Let me write a worker function and run with multiprocessing.

# First, test that mpmath works fine across multiprocessing (it should, since it's pure python).
import multiprocessing as mp
print("mp.get_start_method:", mp.get_start_method())


mp.get_start_method: fork


In [22]:

# Good, fork is available. Let me write the worker as a standalone module/script so it can be imported 
# in subprocesses.

worker_code = '''"""Worker module for computing L_DH zeros in a t-chunk."""
import mpmath
import time

def _setup(dps):
 mpmath.mp.dps = dps
 sqrt5 = mpmath.sqrt(5)
 XI = (mpmath.sqrt(10 - 2*sqrt5) - 2)/(sqrt5 - 1)
 I = mpmath.mpc(0, 1)
 A = (1 - I*XI)/2
 B = (1 + I*XI)/2
 chi = [mpmath.mpc(0), mpmath.mpc(1), I, -I, mpmath.mpc(-1)]
 chib = [mpmath.mpc(0), mpmath.mpc(1), -I, I, mpmath.mpc(-1)]
 log5pi = mpmath.log(5/mpmath.pi)
 return I, A, B, chi, chib, log5pi

def make_Z(dps):
 I, A, B, chi, chib, log5pi = _setup(dps)
 def Z(t):
 t = mpmath.mpf(t)
 s = mpmath.mpc(mpmath.mpf("0.5"), t)
 L1 = mpmath.dirichlet(s, chi)
 L2 = mpmath.dirichlet(s, chib)
 Ls = A*L1 + B*L2
 th = (t/2)*log5pi + mpmath.im(mpmath.loggamma(mpmath.mpc(mpmath.mpf("0.75"), t/2)))
 return mpmath.re(mpmath.exp(I*th)*Ls)
 return Z

def make_L(dps):
 I, A, B, chi, chib, _ = _setup(dps)
 def L(s):
 return A*mpmath.dirichlet(s, chi) + B*mpmath.dirichlet(s, chib)
 return L

def find_zeros_in_chunk(args):
 """Worker function: scan a t-chunk, find sign changes, refine to dps=50.
 Returns list of mpc zeros sorted by Im.
 """
 t_start, t_end, scan_step, scan_dps, refine_dps, max_wall_s = args
 t_start = mpmath.mpf(t_start)
 t_end = mpmath.mpf(t_end)
 scan_step = mpmath.mpf(scan_step)
 deadline = time.time() + max_wall_s if max_wall_s else None

 Z_scan = make_Z(scan_dps)
 # Phase 1: collect sign-change brackets
 brackets = []
 t = t_start
 try:
 z_prev = Z_scan(t)
 except Exception as e:
 return {"error": f"init eval failed at {t}: {e}", "zeros": [], "t_scanned": float(t_start)}
 while t < t_end:
 if deadline and time.time() > deadline:
 break
 t_new = t + scan_step
 if t_new > t_end: t_new = t_end
 try:
 z_new = Z_scan(t_new)
 except Exception:
 t = t_new
 z_prev = None
 continue
 if z_prev is not None and z_prev * z_new < 0:
 brackets.append((t, t_new))
 t = t_new
 z_prev = z_new
 t_scanned = float(t)

 # Phase 2: refine each bracket at dps=refine_dps
 mpmath.mp.dps = refine_dps
 Z_refine = make_Z(refine_dps)
 L_refine = make_L(refine_dps)
 zeros = []
 for (a, b) in brackets:
 if deadline and time.time() > deadline:
 break
 try:
 t0 = mpmath.findroot(Z_refine, (a, b), solver='anderson', tol=mpmath.mpf("1e-45"))
 except Exception:
 try:
 # try bisection fallback
 t0 = mpmath.findroot(Z_refine, (a, b), solver='bisect', tol=mpmath.mpf("1e-40"))
 except Exception:
 continue
 s_zero = mpmath.mpc(mpmath.mpf("0.5"), t0)
 L_val = L_refine(s_zero)
 if abs(L_val) < mpmath.mpf("1e-30"):
 zeros.append(s_zero)
 zeros.sort(key=lambda z: float(mpmath.im(z)))
 return {"zeros": zeros, "t_scanned": t_scanned, "n_brackets": len(brackets), "chunk": (float(t_start), float(t_end))}
'''
from e14c.filesystem import write_file
write_file("/workspace/64235940-e52e-49d9-8fe9-800691f91dac/ldh_worker.py", worker_code)
print("Wrote ldh_worker.py")


Wrote ldh_worker.py


In [23]:

import sys
sys.path.insert(0, '/workspace/64235940-e52e-49d9-8fe9-800691f91dac')
import ldh_worker
import importlib
importlib.reload(ldh_worker)

# Quick test: find zeros in [0, 30] serially
import time
t0 = time.time()
result = ldh_worker.find_zeros_in_chunk((0.5, 30.0, 0.25, 20, 50, 60))
print(f"Time: {time.time()-t0:.1f}s")
print(f"Zeros found: {len(result['zeros'])}, t_scanned: {result['t_scanned']}, brackets: {result['n_brackets']}")
for z in result['zeros'][:5]:
 print(f" {z}")


Time: 4.8s
Zeros found: 10, t_scanned: 30.0, brackets: 10
 (0.5 + 5.0941598445710949256987955170797974750670744531094j)
 (0.5 + 8.9399144080965232447176128773475666256121616213856j)
 (0.5 + 12.133545425800742403702547753765281383126636124091j)
 (0.5 + 14.404003112277498207270097673803851118378453181969j)
 (0.5 + 17.130239400572489600773954527308750679976252956357j)


In [24]:

# Great, the worker works correctly. 4.8s for 10 zeros + scan in [0,30].
# Now let's estimate the full job with parallelization.
#
# Strategy: divide [0, T_max] into chunks. The cost is t-dependent (higher t = slower).
# Use balanced chunks where each chunk takes roughly the same wall time.
# The cost of Z_DH scales roughly like sqrt(t) or t^0.5 (mpmath.dirichlet uses Euler-Maclaurin
# which has cost ~ t^(1/2 + epsilon)). So chunk by equal sqrt(t).
# Chunk i covers t in [T_i, T_{i+1}] where T_i = (i/N)^2 * T_max.

import multiprocessing as mp
import time

# Determine T_max. We want at least 5000 zeros, so go to T_max = 4500 to be safe.
T_MAX = 4500.0
N_CHUNKS = 8 # match CPU count

# Build chunk boundaries: equal sqrt-spacing for balanced load
import math
boundaries = [T_MAX * (i / N_CHUNKS)**2 for i in range(N_CHUNKS + 1)]
boundaries[0] = 0.5 # avoid t=0
print("Chunk boundaries:", [round(b,1) for b in boundaries])
print("Chunk widths:", [round(boundaries[i+1]-boundaries[i],1) for i in range(N_CHUNKS)])

# Each chunk has scan step 0.25, then refines brackets at dps=50.
# Set wall-time budget per chunk to e.g. 2400s (40 min) to leave buffer.
WALL_BUDGET_PER_CHUNK = 2400.0
args = [(boundaries[i], boundaries[i+1], 0.25, 20, 50, WALL_BUDGET_PER_CHUNK) for i in range(N_CHUNKS)]
print("Args ready.")


Chunk boundaries: [0.5, 70.3, 281.2, 632.8, 1125.0, 1757.8, 2531.2, 3445.3, 4500.0]
Chunk widths: [69.8, 210.9, 351.6, 492.2, 632.8, 773.4, 914.1, 1054.7]
Args ready.


In [25]:

# Before launching the full parallel job, let me do a small bench: try the LAST chunk (highest t, slowest)
# but with a short wall budget to see throughput.

import time, multiprocessing as mp
mp.set_start_method('fork', force=True) # safe in jupyter

# Test the heaviest chunk for 60 seconds to estimate throughput
t0 = time.time()
test_result = ldh_worker.find_zeros_in_chunk((4400.0, 4500.0, 0.25, 20, 50, 60.0))
dt = time.time() - t0
print(f"Test chunk (t=4400 to 4500), 60s budget: actual {dt:.1f}s")
print(f" t_scanned to: {test_result['t_scanned']}, n_brackets: {test_result['n_brackets']}, n_zeros: {len(test_result['zeros'])}")


Test chunk (t=4400 to 4500), 60s budget: actual 60.0s
 t_scanned to: 4434.5, n_brackets: 43, n_zeros: 0


In [26]:

# In 60s at t~4400-4500, we only scanned 34 units of t (got 43 brackets) and didn't even begin refinement.
# Scan throughput at t=4400-4500: (4434.5-4400)/60 = 0.575 t-units/sec. At step 0.25, that's 2.3 evals/sec.
# Each Z_DH eval at t=4400, dps=20: 1/2.3 ≈ 435 ms.
# 
# For the highest chunk (t=3445 to 4500, width 1055), at this rate: 1055/0.575 = 1834s just to scan.
# Plus refinement: ~ 1300 brackets * 5s each at dps=50 = 6500s. Total 8334s = 2.3 hours per worker.
# 
# This won't fit in budget. We need MUCH more aggressive optimization.
# 
# Let me implement the fast approximate functional equation for L_DH. This is the only way to get this 
# in budget. ~30 minutes of dev work but ~50-100x speedup at high t.

# Approximate functional equation for L(s, chi), chi primitive mod q, ODD:
# L(s, chi) = sum_{n=1}^X chi(n) n^{-s} + alpha(s,chi) * sum_{n=1}^Y chi_bar(n) n^{s-1} + R(s)
# where alpha(s, chi) = W(chi) * (q/pi)^(1/2 - s) * Gamma((2-s)/2) / Gamma((s+1)/2)
# Choose X = Y = sqrt(qt/(2*pi)) -> balanced
# Truncation error R(s) ~ (X)^{-sigma} for sigma > 0, but for high precision we need to push X up.
# 
# Actually the rigorous version (Hardy-Littlewood) has error ~ O(t^{1/2-sigma}/X^sigma + ...). 
# For dps=50 we need error < 1e-50. We can over-truncate: X = max(K * sqrt(qt/(2pi)), const) with K large.
# 
# For sign-change scanning we only need dps~15. For refinement we need full dps=50.
# 
# Let's prototype:
import mpmath
mpmath.mp.dps = 50

sqrt5 = mpmath.sqrt(5)
XI = (mpmath.sqrt(10 - 2*sqrt5) - 2)/(sqrt5 - 1)
I = mpmath.mpc(0, 1)
A = (1 - I*XI)/2
B = (1 + I*XI)/2
log5pi = mpmath.log(5/mpmath.pi)

# Character values for chi mod 5 (chi(0..4)) and conjugate
chi_vals_lst = [0, 1, I, -I, -1] # period 5
chibar_vals_lst = [0, 1, -I, I, -1] # conjugate

# Gauss sum & root number
tau_chi = sum(chi_vals_lst[a] * mpmath.exp(2*mpmath.pi*I*a/5) for a in range(1, 5))
tau_chibar = sum(chibar_vals_lst[a] * mpmath.exp(2*mpmath.pi*I*a/5) for a in range(1, 5))
# For odd chi: W(chi) = tau(chi) / (i * sqrt(q))
W_chi = tau_chi / (I * mpmath.sqrt(5))
W_chibar = tau_chibar / (I * mpmath.sqrt(5))
print("W(chi):", W_chi, "|W|:", abs(W_chi))
print("W(chibar):", W_chibar, "|W|:", abs(W_chibar))

def alpha_L(s, W, q=5):
 """Multiplier in approximate functional equation: alpha such that 
 L(s, chi) ≈ S_main(s) + alpha * S_dual(s) with S_dual using chi_bar.
 alpha(s) = W * (q/pi)^(1/2 - s) * Gamma((2-s)/2)/Gamma((s+1)/2)
 """
 return W * (mpmath.mpf(q)/mpmath.pi)**(mpmath.mpf("0.5") - s) * mpmath.gamma((2-s)/2) / mpmath.gamma((s+1)/2)

def L_chi_AFE(s, chi_vals, chibar_vals, W, q=5, K=4.0):
 """Approximate functional equation for L(s, chi) on (or near) critical line, t > 0.
 K is over-truncation factor. Larger K -> more accuracy.
 """
 t = float(mpmath.im(s))
 X_real = max(1.0, (q * abs(t) / (2 * 3.14159265358979)) ** 0.5)
 X = int(K * X_real) + 5
 # Main sum: sum_{n=1}^X chi(n) / n^s
 S_main = mpmath.mpc(0)
 for n in range(1, X+1):
 c = chi_vals[n % 5]
 if c != 0:
 S_main += c * mpmath.power(n, -s)
 # Dual sum: sum_{n=1}^X chi_bar(n) / n^(1-s)
 S_dual = mpmath.mpc(0)
 one_minus_s = 1 - s
 for n in range(1, X+1):
 c = chibar_vals[n % 5]
 if c != 0:
 S_dual += c * mpmath.power(n, -one_minus_s)
 a = alpha_L(s, W, q)
 return S_main + a * S_dual

def L_DH_AFE(s, K=4.0):
 """Approximate L_DH via AFE on both L_chi and L_chibar."""
 L1 = L_chi_AFE(s, chi_vals_lst, chibar_vals_lst, W_chi, 5, K)
 L2 = L_chi_AFE(s, chibar_vals_lst, chi_vals_lst, W_chibar, 5, K)
 return A * L1 + B * L2

# Verify against mpmath.dirichlet at a few points:
s_test = mpmath.mpc(mpmath.mpf("0.5"), mpmath.mpf("100"))
v_afe = L_DH_AFE(s_test, K=4.0)
chi_l = [mpmath.mpc(0), mpmath.mpc(1), I, -I, mpmath.mpc(-1)]
chib_l = [mpmath.mpc(0), mpmath.mpc(1), -I, I, mpmath.mpc(-1)]
v_ref = A * mpmath.dirichlet(s_test, chi_l) + B * mpmath.dirichlet(s_test, chib_l)
print(f"At s=1/2+100i:")
print(f" AFE: {v_afe}")
print(f" Ref: {v_ref}")
print(f" diff: {abs(v_afe - v_ref)}")


W(chi): (0.85065080835203993218154049706301107224040140376482 + 0.52573111211913360602566908484787660728549793224334j) |W|: 1.0
W(chibar): (0.85065080835203993218154049706301107224040140376482 - 0.52573111211913360602566908484787660728549793224334j) |W|: 1.0
At s=1/2+100i:
 AFE: (-0.19536900910334989505097445419844880702286663434237 - 0.086437897808901240474763029876596197978602003959283j)
 Ref: (0.79616859587664123777450189779670526105936924335843 + 0.35225207951296046856243371921620871664345543278316j)
 diff: 1.0842489189720746394853689343825389158038808926729


In [27]:

# Big disagreement — my AFE has a sign / coefficient error. Let me re-derive carefully.
#
# Functional equation for L(s, chi), chi primitive mod q, odd (chi(-1) = -1):
# Lambda(s, chi) = (q/pi)^((s+1)/2) * Gamma((s+1)/2) * L(s, chi)
# (Some references use (s+1)/2 in the prefactor, others use s/2 — be careful.)
#
# More precisely (Davenport "Multiplicative Number Theory" Ch.9):
# For chi mod q, primitive, define
# xi(s, chi) = (q/pi)^((s+a)/2) Gamma((s+a)/2) L(s, chi), where a=0 (even) or a=1 (odd).
# Then xi(s, chi) = W(chi) * xi(1-s, chi_bar), where W(chi) = tau(chi) / (i^a sqrt(q)).
#
# So for odd chi mod 5, a=1, W(chi) = tau(chi)/(i*sqrt(5)). OK.
# 
# Solving for L(s, chi):
# L(s, chi) = W(chi) * (q/pi)^((1-s+1)/2 - (s+1)/2) * Gamma((2-s)/2)/Gamma((s+1)/2) * L(1-s, chi_bar)
# = W(chi) * (q/pi)^((1-2s)/2 + 0) wait let's compute exponent: 
# (q/pi)^((1-s+1)/2) / (q/pi)^((s+1)/2) = (q/pi)^((2-s)/2 - (s+1)/2) = (q/pi)^((1-2s)/2) = (q/pi)^(1/2 - s)
# Good. So:
# L(s, chi) = W(chi) * (q/pi)^(1/2 - s) * Gamma((2-s)/2)/Gamma((s+1)/2) * L(1-s, chi_bar)
# 
# That's what I had. So the alpha is right.
#
# The error must be in the AFE truncation. The simple AFE
# L(s, chi) = sum_{n<=X} chi(n)/n^s + alpha * sum_{n<=Y} chi_bar(n)/n^(1-s)
# has a NON-NEGLIGIBLE error term. With X = Y = sqrt(qt/(2pi)) the error is only O(t^{-1/4}).
# That's only ~10 ms when t=100... wait, t^{-1/4} = 100^{-0.25} = 0.316. NOT small!
# 
# The classical AFE has O(t^{-1/4}) error, NOT good enough even at moderate t. We need the FULL 
# Riemann-Siegel formula with asymptotic corrections, which is significantly more involved.
#
# This is becoming too complex. Let me step back and make a pragmatic decision.
print("Re-strategizing.")


Re-strategizing.


In [28]:

# Wait — actually with K=4 (over-truncation factor 4), the error should be MUCH smaller than O(t^{-1/4}). 
# The diff is 1.08 — that's the size of the L-values themselves! So something is fundamentally wrong, 
# not just truncation error.
#
# Let me re-derive more carefully. Hardy-Littlewood AFE: for s = sigma + it, t > 0,
# L(s, chi) = sum_{n<=X} chi(n)/n^s + chi_factor(s) * sum_{n<=Y} chi_bar(n)/n^(1-s) + error
# with XY = qt/(2*pi). 
# chi_factor(s) here is the alpha. With X = Y = sqrt(qt/(2pi)) the error is O(X^{1/2-sigma} log) 
# or similar — not O(t^{-1/4}) as I said.
#
# Let me check via mpmath at s = 0.5 + 1000i with a simple L mod 5:
mpmath.mp.dps = 30
s_t = mpmath.mpc(mpmath.mpf("0.5"), mpmath.mpf("1000"))
L_ref = mpmath.dirichlet(s_t, chi_l)

# Manual AFE for L(s, chi):
import math
q = 5
t_val = float(mpmath.im(s_t))
Xbal = math.sqrt(q * t_val / (2 * math.pi))
X = int(2 * Xbal) + 10 # K=2

S1 = mpmath.mpc(0)
for n in range(1, X+1):
 c = chi_vals_lst[n % 5]
 if c != 0:
 S1 += c * mpmath.power(n, -s_t)
S2 = mpmath.mpc(0)
for n in range(1, X+1):
 c = chibar_vals_lst[n % 5]
 if c != 0:
 S2 += c * mpmath.power(n, -(1-s_t))

a = W_chi * (mpmath.mpf(q)/mpmath.pi)**(mpmath.mpf("0.5") - s_t) * mpmath.gamma((2-s_t)/2) / mpmath.gamma((s_t+1)/2)
L_afe = S1 + a * S2
print(f"X = {X}, Xbal = {Xbal:.1f}")
print(f"L_ref (mpmath.dirichlet): {L_ref}")
print(f"L_afe: {L_afe}")
print(f"diff: {abs(L_afe - L_ref)}")


X = 66, Xbal = 28.2
L_ref (mpmath.dirichlet): (0.725055694521126385337268013555 + 0.166530152180958709707242663066j)
L_afe: (-2.10311910976222792349896350102 - 0.483042541490143799644801404259j)
diff: 2.90181277961661775913268003169


In [29]:

# The AFE is fundamentally wrong — even with X = 2*sqrt(qt/2pi) = 66 we get diff of order |L| ~1.
# So my AFE formulation has a bug.
#
# Let me check by INCREASING X massively — if AFE is correct, larger X must converge.

X_large = 1000
S1 = mpmath.mpc(0)
for n in range(1, X_large+1):
 c = chi_vals_lst[n % 5]
 if c != 0:
 S1 += c * mpmath.power(n, -s_t)
S2 = mpmath.mpc(0)
for n in range(1, X_large+1):
 c = chibar_vals_lst[n % 5]
 if c != 0:
 S2 += c * mpmath.power(n, -(1-s_t))
L_afe = S1 + a * S2
print(f"X = {X_large}")
print(f"L_ref: {L_ref}")
print(f"L_afe: {L_afe}")
print(f"diff: {abs(L_afe - L_ref)}")


X = 1000
L_ref: (0.725055694521126385337268013555 + 0.166530152180958709707242663066j)
L_afe: (1.52851303008171524574571567038 + 0.351067524099929085096663065218j)
diff: 0.824377178056792814918847264333


In [30]:

# Still bad. This shows the dual sum at large X doesn't go to zero — it diverges since Re(1-s) = 1/2 (boundary).
# The approximate functional equation is more subtle. The two sums must be truncated with proper smoothing.
# 
# The classical AFE (Hardy-Littlewood, 1923) for the Riemann zeta on the critical line:
# zeta(s) = sum_{n<=X} 1/n^s + chi_zeta(s) sum_{n<=Y} 1/n^(1-s) + R
# is valid only with proper choice and has POLYNOMIAL error in t. The clean Riemann-Siegel formula 
# requires the asymptotic expansion of the remainder.
#
# I clearly don't have time to implement this from scratch correctly in this notebook.
# 
# DECISION: Go back to using mpmath.dirichlet (slow but correct). Run as many zeros as possible in 
# parallel and save what we get. Document the limitation clearly.

# Let me re-plan:
# Per-chunk wall budget: 55 minutes total budget across all 8 cores (with some margin).
# We've used ~13 minutes already. Remaining: ~45 minutes wallclock.
# 
# But within Jupyter, mp.Pool runs across cores so we get 8x more compute. 45 min * 8 cores = 360 CPU-min.
# At avg 250 ms/eval, that's ~86400 evals total. 
# 
# Cost breakdown for scanning [0.5, T_max] step 0.25 at dps=20:
# - At t~100: 40 ms/eval
# - At t~1000: 150 ms/eval 
# - At t~3000: 350 ms/eval
# - At t~4500: 450 ms/eval
# Avg ~ 250 ms/eval. Scan needs 4500/0.25 = 18000 evals = ~75 min on 1 core, but parallelized 8x: ~10 min.
# 
# Refinement at dps=50: each zero takes ~10 evals * 500 ms = 5s/zero serial.
# 5000 zeros * 5s = 25000s on 1 core = 7 hours. With 8 cores: ~52 min. Tight but feasible.
# 
# Total parallel: ~62 min. We have ~45 remaining. Won't quite fit at full dps=50.
# 
# Compromise: Refine at dps=30 (sufficient for many decimal places, but doesn't meet spec).
# OR: refine at dps=50 but use a tighter findroot (Newton from middle of bracket, ~5 evals instead of ~10).
# 
# Let me just start running with dps=50 refinement in parallel and accept partial completion if needed.
# I'll explicitly document this.

print("Launching parallel computation with dps=50 refinement.")
print("Budget tight — may yield partial result.")


Launching parallel computation with dps=50 refinement.
Budget tight — may yield partial result.


In [31]:

# One more optimization to try: mpmath.findroot using Newton's method requires derivative or finite diff.
# Default solver for bracketed problems is 'anderson'. Let me check whether 'secant' or 'illinois' is faster.

import time
mpmath.mp.dps = 50
Z50_local = make_Z_DH(50)

# Time anderson vs others
br = (mpmath.mpf("2000.5"), mpmath.mpf("2000.75"))
for solver in ['anderson', 'bisect', 'illinois', 'pegasus', 'ridder']:
 t0 = time.time()
 try:
 z = mpmath.findroot(Z50_local, br, solver=solver, tol=mpmath.mpf("1e-45"))
 print(f" {solver}: {time.time()-t0:.2f}s -> t={z}")
 except Exception as e:
 print(f" {solver}: FAILED ({e})")


 anderson: 4.66s -> t=2000.6831188525078987549353563568788029657328556907


 bisect: 43.22s -> t=2000.6831188525078987549353563568787286788750421621


 illinois: 6.00s -> t=2000.6831188525078987549353563568788029657328556907


 pegasus: 4.55s -> t=2000.6831188525078987549353563568788029657328556907


 ridder: FAILED (Could not find root within given tolerance. (0.34297316127020374880314466950700238831518414796173808228 > 9.9999999999999999999999999999999999999999999999999963954e-46)
Try another starting point or tweak arguments.)


In [32]:

# pegasus is fastest at 4.55s. ~10 Z_DH evaluations per refinement.
# 
# Key insight: refine after the bracket is tight. Use 'pegasus' solver.
# 
# Update the worker file accordingly:
worker_code_v2 = '''"""Worker module for computing L_DH zeros in a t-chunk (v2 — pegasus solver)."""
import mpmath
import time

def _setup(dps):
 mpmath.mp.dps = dps
 sqrt5 = mpmath.sqrt(5)
 XI = (mpmath.sqrt(10 - 2*sqrt5) - 2)/(sqrt5 - 1)
 I = mpmath.mpc(0, 1)
 A = (1 - I*XI)/2
 B = (1 + I*XI)/2
 chi = [mpmath.mpc(0), mpmath.mpc(1), I, -I, mpmath.mpc(-1)]
 chib = [mpmath.mpc(0), mpmath.mpc(1), -I, I, mpmath.mpc(-1)]
 log5pi = mpmath.log(5/mpmath.pi)
 return I, A, B, chi, chib, log5pi

def make_Z(dps):
 I, A, B, chi, chib, log5pi = _setup(dps)
 def Z(t):
 t = mpmath.mpf(t)
 s = mpmath.mpc(mpmath.mpf("0.5"), t)
 L1 = mpmath.dirichlet(s, chi)
 L2 = mpmath.dirichlet(s, chib)
 Ls = A*L1 + B*L2
 th = (t/2)*log5pi + mpmath.im(mpmath.loggamma(mpmath.mpc(mpmath.mpf("0.75"), t/2)))
 return mpmath.re(mpmath.exp(I*th)*Ls)
 return Z

def make_L(dps):
 I, A, B, chi, chib, _ = _setup(dps)
 def L(s):
 return A*mpmath.dirichlet(s, chi) + B*mpmath.dirichlet(s, chib)
 return L

def find_zeros_in_chunk(args):
 """args = (t_start, t_end, scan_step, scan_dps, refine_dps, max_wall_s, save_path)
 Periodically save partial results to save_path (pickle of (zeros, t_done, n_brackets_remaining)).
 """
 import pickle
 t_start, t_end, scan_step, scan_dps, refine_dps, max_wall_s, save_path = args
 t_start = mpmath.mpf(t_start)
 t_end = mpmath.mpf(t_end)
 scan_step = mpmath.mpf(scan_step)
 deadline = time.time() + max_wall_s if max_wall_s else None

 # Phase 1: scan brackets
 Z_scan = make_Z(scan_dps)
 brackets = []
 t = t_start
 try:
 z_prev = Z_scan(t)
 except Exception as e:
 return {"error": f"init eval failed at {t}: {e}", "zeros": [], "t_scanned": float(t_start)}
 while t < t_end:
 if deadline and time.time() > deadline:
 break
 t_new = t + scan_step
 if t_new > t_end:
 t_new = t_end
 try:
 z_new = Z_scan(t_new)
 except Exception:
 t = t_new
 z_prev = None
 continue
 if z_prev is not None and z_prev * z_new < 0:
 brackets.append((t, t_new))
 t = t_new
 z_prev = z_new
 t_scanned = float(t)

 # Phase 2: refine
 mpmath.mp.dps = refine_dps
 Z_ref = make_Z(refine_dps)
 L_ref = make_L(refine_dps)
 zeros = []
 last_save = time.time()
 for idx, (a, b) in enumerate(brackets):
 if deadline and time.time() > deadline:
 break
 try:
 t0 = mpmath.findroot(Z_ref, (a, b), solver='pegasus', tol=mpmath.mpf("1e-45"))
 except Exception:
 try:
 t0 = mpmath.findroot(Z_ref, (a, b), solver='anderson', tol=mpmath.mpf("1e-40"))
 except Exception:
 continue
 s_zero = mpmath.mpc(mpmath.mpf("0.5"), t0)
 L_val = L_ref(s_zero)
 if abs(L_val) < mpmath.mpf("1e-30"):
 zeros.append(s_zero)
 # periodically checkpoint
 if save_path and (time.time() - last_save > 60):
 try:
 with open(save_path, "wb") as f:
 pickle.dump({"zeros": zeros, "n_brackets": len(brackets), "idx": idx, "t_scanned": t_scanned}, f)
 last_save = time.time()
 except Exception:
 pass

 zeros.sort(key=lambda z: float(mpmath.im(z)))
 result = {"zeros": zeros, "t_scanned": t_scanned, "n_brackets": len(brackets), "chunk": (float(t_start), float(t_end))}
 if save_path:
 try:
 with open(save_path, "wb") as f:
 pickle.dump(result, f)
 except Exception:
 pass
 return result
'''
from e14c.filesystem import write_file
write_file("/workspace/64235940-e52e-49d9-8fe9-800691f91dac/ldh_worker.py", worker_code_v2)
print("Wrote ldh_worker.py v2")


Wrote ldh_worker.py v2


In [33]:

# Plan to launch the parallel job. 
# 
# To budget: We've used ~17 minutes of the 60 minute limit. Remaining ~43 minutes.
# Set per-chunk wall budget to 2200s = 36.7 min (with buffer for collection).
#
# Reset modules in subprocess context: in fork, the worker module needs to be importable.
# Let's set up using multiprocessing.

import os, sys, time
os.chdir("/workspace/64235940-e52e-49d9-8fe9-800691f91dac")
sys.path.insert(0, "/workspace/64235940-e52e-49d9-8fe9-800691f91dac")

# Important: reload the worker module so subprocesses get v2
if 'ldh_worker' in sys.modules:
 del sys.modules['ldh_worker']

# Re-determine T_max and chunk layout.
# We want first 5000 zeros. By N(T) formula, need T_max ~ 4400. Set to 4600 for safety margin.
# However: the highest chunk takes longest. Better to allocate more cores to high-t chunks.
# 
# Strategy: 12 chunks with sqrt-spacing, run 8 at a time (Pool of 8). Excess will queue.
# Actually since Pool keeps cores busy, having more chunks than cores is fine.

T_MAX = 4600.0
N_CHUNKS = 12
boundaries = [T_MAX * (i / N_CHUNKS)**1.7 for i in range(N_CHUNKS + 1)] # less aggressive than ^2
boundaries[0] = 0.5
print("Chunk boundaries:", [round(b,1) for b in boundaries])

WALL_PER_CHUNK = 2000.0 # 33 minutes each (will run partly in parallel)

# Save paths for checkpointing
save_dir = "/workspace/64235940-e52e-49d9-8fe9-800691f91dac/ldh_chunks"
os.makedirs(save_dir, exist_ok=True)

args_list = []
for i in range(N_CHUNKS):
 args_list.append((boundaries[i], boundaries[i+1], 0.25, 20, 50, WALL_PER_CHUNK, 
 f"{save_dir}/chunk_{i:02d}.pkl"))
print(f"Will spawn {N_CHUNKS} chunk tasks across 8 cores.")


Chunk boundaries: [0.5, 67.3, 218.7, 435.8, 710.6, 1038.5, 1415.8, 1840.0, 2308.9, 2820.7, 3374.0, 3967.5, 4600.0]
Will spawn 12 chunk tasks across 8 cores.


In [34]:

# Launch parallel computation. Use multiprocessing.Pool with apply_async so we can monitor.
import multiprocessing as mp
import time, pickle

# Use spawn to be safe in jupyter, but we need to make sure ldh_worker is importable from spawn.
# Stick with fork — it inherits process state which is what we want.
mp.set_start_method('fork', force=True)

# Use an explicit Pool and apply_async
import ldh_worker
import importlib
importlib.reload(ldh_worker)

start_wall = time.time()
print(f"Launching at wallclock {time.strftime('%H:%M:%S')}")

pool = mp.Pool(processes=8)
async_results = [pool.apply_async(ldh_worker.find_zeros_in_chunk, (args,)) for args in args_list]
pool.close()
print(f"Pool started, {len(async_results)} chunks submitted.")


Launching at wallclock 05:01:31


Pool started, 12 chunks submitted.


In [35]:

# Monitor progress. Poll status periodically.
import time, os, pickle

def status():
 print(f"Wallclock elapsed: {time.time()-start_wall:.0f}s")
 n_done = sum(r.ready() for r in async_results)
 print(f" Chunks ready: {n_done}/{len(async_results)}")
 for i, r in enumerate(async_results):
 if r.ready():
 try:
 res = r.get(timeout=1)
 if isinstance(res, dict) and 'zeros' in res:
 print(f" Chunk {i:02d} [{args_list[i][0]:.1f}, {args_list[i][1]:.1f}]: {len(res['zeros'])} zeros, t_scan={res['t_scanned']:.1f}, brackets={res.get('n_brackets','?')}")
 else:
 print(f" Chunk {i:02d}: result keys: {res}")
 except Exception as e:
 print(f" Chunk {i:02d}: error {e}")
 # peek at partial checkpoints for non-done chunks
 for i, r in enumerate(async_results):
 if not r.ready():
 ckpt = f"/workspace/64235940-e52e-49d9-8fe9-800691f91dac/ldh_chunks/chunk_{i:02d}.pkl"
 if os.path.exists(ckpt):
 try:
 with open(ckpt, 'rb') as f:
 ck = pickle.load(f)
 nz = len(ck.get('zeros', [])) if ck.get('zeros') else 0
 idx = ck.get('idx', '?')
 nb = ck.get('n_brackets', '?')
 print(f" Chunk {i:02d} [{args_list[i][0]:.1f},{args_list[i][1]:.1f}] (running): {nz} zeros refined so far, idx={idx}/{nb}, t_scanned={ck.get('t_scanned','?')}")
 except Exception:
 print(f" Chunk {i:02d}: checkpoint unreadable yet")

status()


Wallclock elapsed: 14s
 Chunks ready: 0/12


In [36]:

# Wait a bit before checking
time.sleep(90)
status()


Wallclock elapsed: 109s
 Chunks ready: 1/12
 Chunk 00 [0.5, 67.3]: 32 zeros, t_scan=67.3, brackets=32
 Chunk 01 [67.3,218.7] (running): 57 zeros refined so far, idx=56/105, t_scanned=218.7261487023512


In [37]:

time.sleep(180)
status()


Wallclock elapsed: 294s
 Chunks ready: 2/12
 Chunk 00 [0.5, 67.3]: 32 zeros, t_scan=67.3, brackets=32
 Chunk 01 [67.3, 218.7]: 105 zeros, t_scan=218.7, brackets=105
 Chunk 02 [218.7,435.8] (running): 65 zeros refined so far, idx=64/181, t_scanned=435.76851287173946
 Chunk 03 [435.8,710.6] (running): 21 zeros refined so far, idx=20/244, t_scanned=710.6433537170203


In [38]:

# Note: only chunks 02 and 03 are showing progress so far among the running ones. The other chunks 
# (04-11) haven't checkpointed yet — they're either still scanning or have completed scan and started 
# refining but haven't hit a 60s checkpoint interval.
# 
# Important observation: Chunk 00 (0.5 to 67.3) found only 32 zeros — that's the density there. 
# But we'd expect ~37 zeros given N(67.3) ≈ (67.3/(2π)) log(5*67.3/(2πe)) = 10.71*log(19.7) = 31.9.
# Close to predicted. Good.
# 
# Chunk 01 (67.3 to 218.7) found 105 zeros. N(218.7) - N(67.3) ≈ 132 - 32 = 100. Match!

# Wait more
time.sleep(180)
status()


Wallclock elapsed: 483s
 Chunks ready: 2/12
 Chunk 00 [0.5, 67.3]: 32 zeros, t_scan=67.3, brackets=32
 Chunk 01 [67.3, 218.7]: 105 zeros, t_scan=218.7, brackets=105
 Chunk 02 [218.7,435.8] (running): 127 zeros refined so far, idx=126/181, t_scanned=435.76851287173946
 Chunk 03 [435.8,710.6] (running): 80 zeros refined so far, idx=79/244, t_scanned=710.6433537170203
 Chunk 04 [710.6,1038.5] (running): 25 zeros refined so far, idx=24/315, t_scanned=1038.4813961239147


In [39]:

# Progress is being made. Chunk 04 started refining around 7-8 minutes in.
# Let me wait more and check.
time.sleep(300)
status()


Wallclock elapsed: 788s
 Chunks ready: 3/12
 Chunk 00 [0.5, 67.3]: 32 zeros, t_scan=67.3, brackets=32
 Chunk 01 [67.3, 218.7]: 105 zeros, t_scan=218.7, brackets=105
 Chunk 02 [218.7, 435.8]: 181 zeros, t_scan=435.8, brackets=181
 Chunk 03 [435.8,710.6] (running): 168 zeros refined so far, idx=167/244, t_scanned=710.6433537170203
 Chunk 04 [710.6,1038.5] (running): 87 zeros refined so far, idx=86/315, t_scanned=1038.4813961239147
 Chunk 05 [1038.5,1415.8] (running): 36 zeros refined so far, idx=35/375, t_scanned=1415.8160753466539
 Chunk 06 [1415.8,1840.0] (running): 8 zeros refined so far, idx=7/430, t_scanned=1839.9938923667373


In [40]:

# Steady progress. Let me make some projections:
# Chunk 03 (~280 zeros over t in 435-710): at 168 done, refinement rate ~ 168/(788-294) = ~0.34 zeros/sec
# Chunk 06 just started refining, ~430 zeros to do at the low-rate end... will take ~430/0.34 = ~21 min more.
# 
# Let me also check how many cores are being used. All 8 should be busy. Looks good — chunks 02, 03, 04, 05, 06 
# are running plus the higher chunks (07-11) likely also running.

time.sleep(360)
status()


Wallclock elapsed: 1156s
 Chunks ready: 3/12
 Chunk 00 [0.5, 67.3]: 32 zeros, t_scan=67.3, brackets=32
 Chunk 01 [67.3, 218.7]: 105 zeros, t_scan=218.7, brackets=105
 Chunk 02 [218.7, 435.8]: 181 zeros, t_scan=435.8, brackets=181
 Chunk 03 [435.8,710.6] (running): 232 zeros refined so far, idx=231/244, t_scanned=710.6433537170203
 Chunk 04 [710.6,1038.5] (running): 159 zeros refined so far, idx=158/315, t_scanned=1038.4813961239147
 Chunk 05 [1038.5,1415.8] (running): 107 zeros refined so far, idx=106/375, t_scanned=1415.8160753466539
 Chunk 06 [1415.8,1840.0] (running): 45 zeros refined so far, idx=44/430, t_scanned=1839.9938923667373
 Chunk 07 [1840.0,2308.9] (running): 22 zeros refined so far, idx=21/495, t_scanned=2308.8870680451264


In [41]:

# Good. Chunks 03 nearly done. 07 just started.
# By extrapolation: Chunk 06 (430 brackets) won't finish for another ~10 min. Chunk 07 ~20 min. 
# Higher chunks (08-11) probably won't finish in time.
# 
# We are at 1156s wallclock from process start; we have ~2400s = 40 min remaining budget.
# 
# Critical: at this point I should pre-emptively gather what we have. Even if some chunks don't 
# fully complete, the checkpointing means we have partial results.
# 
# Let's also think about chunks 08-11: those haven't even started refining yet (they're still scanning, 
# and each scan needs a single core for a long time at high t). Their per-chunk wall budget is 2000s 
# (~33 min). They started at process launch, so they'll have ~2000s of wall time to finish their scan 
# + as much refinement as they can.
#
# Continue monitoring.
time.sleep(360)
status()


Wallclock elapsed: 1527s
 Chunks ready: 4/12
 Chunk 00 [0.5, 67.3]: 32 zeros, t_scan=67.3, brackets=32
 Chunk 01 [67.3, 218.7]: 105 zeros, t_scan=218.7, brackets=105
 Chunk 02 [218.7, 435.8]: 181 zeros, t_scan=435.8, brackets=181
 Chunk 03 [435.8, 710.6]: 244 zeros, t_scan=710.6, brackets=244
 Chunk 04 [710.6,1038.5] (running): 231 zeros refined so far, idx=230/315, t_scanned=1038.4813961239147
 Chunk 05 [1038.5,1415.8] (running): 173 zeros refined so far, idx=172/375, t_scanned=1415.8160753466539
 Chunk 06 [1415.8,1840.0] (running): 88 zeros refined so far, idx=87/430, t_scanned=1839.9938923667373
 Chunk 07 [1840.0,2308.9] (running): 66 zeros refined so far, idx=65/495, t_scanned=2308.8870680451264


In [42]:

# Status: ~25.4 min wall elapsed. Still no checkpoints from chunks 08-11.
# Probably they're still scanning (high t = slow Z_DH evals).
# Chunks finished: 00, 01, 02, 03 (562 zeros for t in [0.5, 710.6])
# 
# Total zeros found so far (including running): 32+105+181+244+231+173+88+66 = 1120 zeros.
# 
# Let me check chunks 08-11 directly via their checkpoints (they may not have hit the 60s save yet)
import os
for i in range(8, 12):
 ckpt = f"/workspace/64235940-e52e-49d9-8fe9-800691f91dac/ldh_chunks/chunk_{i:02d}.pkl"
 print(f"chunk_{i:02d}: exists={os.path.exists(ckpt)}", end="")
 if os.path.exists(ckpt):
 try:
 with open(ckpt, 'rb') as f:
 ck = pickle.load(f)
 print(f" zeros={len(ck.get('zeros',[]))}, idx={ck.get('idx','?')}, n_brackets={ck.get('n_brackets','?')}, t_scanned={ck.get('t_scanned','?')}")
 except Exception as e:
 print(f" err: {e}")
 else:
 print()


chunk_08: exists=False
chunk_09: exists=False
chunk_10: exists=False
chunk_11: exists=False


In [43]:

# Chunks 8-11 still scanning (haven't even completed scan phase, so no save yet — checkpoint 
# happens only during refinement). They cover t in [2308.9, 4600]. The scan throughput at t~2500-4500 
# is very slow.
#
# Given the time budget, chunks 08-11 will likely NOT contribute zeros within 2000s wall budget.
# I should consider trimming our scope: focus on getting as many zeros as possible from chunks 0-7,
# which target up to T ≈ 2309. By N(T) formula: N(2309) ~ (2309/2π)·log(5·2309/(2πe)) = 367.5·log(675.6) 
# = 367.5·6.516 = 2395 zeros. So chunks 0-7 (if they complete) give ~2395 zeros.
# 
# Continue monitoring.
time.sleep(360)
status()


Wallclock elapsed: 1918s
 Chunks ready: 4/12
 Chunk 00 [0.5, 67.3]: 32 zeros, t_scan=67.3, brackets=32
 Chunk 01 [67.3, 218.7]: 105 zeros, t_scan=218.7, brackets=105
 Chunk 02 [218.7, 435.8]: 181 zeros, t_scan=435.8, brackets=181
 Chunk 03 [435.8, 710.6]: 244 zeros, t_scan=710.6, brackets=244
 Chunk 04 [710.6,1038.5] (running): 304 zeros refined so far, idx=303/315, t_scanned=1038.4813961239147
 Chunk 05 [1038.5,1415.8] (running): 225 zeros refined so far, idx=224/375, t_scanned=1415.8160753466539
 Chunk 06 [1415.8,1840.0] (running): 133 zeros refined so far, idx=132/430, t_scanned=1839.9938923667373
 Chunk 07 [1840.0,2308.9] (running): 112 zeros refined so far, idx=111/495, t_scanned=2308.8870680451264
 Chunk 08 [2308.9,2820.7] (running): 35 zeros refined so far, idx=34/568, t_scanned=2820.733000210696
 Chunk 09 [2820.7,3374.0] (running): 5 zeros refined so far, idx=4/626, t_scanned=3374.036010292075


In [44]:

# We have ~14 minutes left. I need to terminate the pool, gather all available results, save them, 
# and submit. Let me wait a bit more for any quick refinements then collect.

time.sleep(180)
status()


Wallclock elapsed: 2105s
 Chunks ready: 9/12
 Chunk 00 [0.5, 67.3]: 32 zeros, t_scan=67.3, brackets=32
 Chunk 01 [67.3, 218.7]: 105 zeros, t_scan=218.7, brackets=105
 Chunk 02 [218.7, 435.8]: 181 zeros, t_scan=435.8, brackets=181
 Chunk 03 [435.8, 710.6]: 244 zeros, t_scan=710.6, brackets=244
 Chunk 04 [710.6, 1038.5]: 315 zeros, t_scan=1038.5, brackets=315
 Chunk 05 [1038.5, 1415.8]: 239 zeros, t_scan=1415.8, brackets=375
 Chunk 06 [1415.8, 1840.0]: 152 zeros, t_scan=1840.0, brackets=430
 Chunk 07 [1840.0, 2308.9]: 127 zeros, t_scan=2308.9, brackets=495
 Chunk 08 [2308.9, 2820.7]: 54 zeros, t_scan=2820.7, brackets=568
 Chunk 09 [2820.7,3374.0] (running): 23 zeros refined so far, idx=22/626, t_scanned=3374.036010292075


In [45]:

# 9 chunks finished - they hit their wall budget. Need to terminate pool NOW and collect.
# 
# Several chunks (05-09) hit wall budget before completing all refinements:
# Chunk 05: 239/375 done (rest unrefined brackets)
# Chunk 06: 152/430 done
# Chunk 07: 127/495 done
# Chunk 08: 54/568 done 
# Chunk 09: 23/626 done (still running)
# 
# This is partial — we have ~1449 zeros refined so far in the contiguous range, plus chunks 05-09 
# have gaps. We need to be HONEST that we have gaps.
# 
# Actually: chunk 00-04 are COMPLETE: 32+105+181+244+315 = 877 zeros, contiguous, covering t in [0.5, 1038.5].
# Chunks 05-09 are PARTIAL — they refined SOME brackets but not all, with gaps in coverage.
#
# Let me terminate pool and gather all results immediately.

pool.terminate()
pool.join()
print("Pool terminated.")

# Gather results from all available checkpoints
import pickle, os
all_zeros = []
chunk_info = []
for i in range(12):
 ckpt = f"/workspace/64235940-e52e-49d9-8fe9-800691f91dac/ldh_chunks/chunk_{i:02d}.pkl"
 if os.path.exists(ckpt):
 try:
 with open(ckpt, 'rb') as f:
 ck = pickle.load(f)
 zeros = ck.get('zeros', [])
 all_zeros.extend(zeros)
 chunk_info.append({
 "chunk": i,
 "t_range": (args_list[i][0], args_list[i][1]),
 "n_zeros": len(zeros),
 "n_brackets": ck.get('n_brackets', None),
 "t_scanned": ck.get('t_scanned', None),
 "complete": (ck.get('idx', -1) is None) or (ck.get('idx', 0) + 1 >= ck.get('n_brackets', 0) if ck.get('n_brackets') else False),
 })
 print(f"Chunk {i}: {len(zeros)} zeros, brackets={ck.get('n_brackets')}, idx={ck.get('idx')}")
 except Exception as e:
 print(f"Chunk {i}: error reading checkpoint: {e}")
 else:
 # If pool returned result, use it
 try:
 if async_results[i].ready():
 res = async_results[i].get(timeout=1)
 zeros = res.get('zeros', [])
 all_zeros.extend(zeros)
 chunk_info.append({
 "chunk": i, "t_range": (args_list[i][0], args_list[i][1]),
 "n_zeros": len(zeros), "n_brackets": res.get('n_brackets'),
 "t_scanned": res.get('t_scanned'), "complete": True,
 })
 print(f"Chunk {i}: (from pool) {len(zeros)} zeros")
 else:
 print(f"Chunk {i}: no checkpoint, not ready")
 except Exception as e:
 print(f"Chunk {i}: {e}")

print(f"\nTotal zeros collected: {len(all_zeros)}")


Pool terminated.
Chunk 0: 32 zeros, brackets=32, idx=None
Chunk 1: 105 zeros, brackets=105, idx=None
Chunk 2: 181 zeros, brackets=181, idx=None
Chunk 3: 244 zeros, brackets=244, idx=None
Chunk 4: 315 zeros, brackets=315, idx=None
Chunk 5: 239 zeros, brackets=375, idx=None
Chunk 6: 152 zeros, brackets=430, idx=None
Chunk 7: 127 zeros, brackets=495, idx=None
Chunk 8: 54 zeros, brackets=568, idx=None
Chunk 9: 23 zeros, brackets=626, idx=22
Chunk 10: no checkpoint, not ready
Chunk 11: no checkpoint, not ready

Total zeros collected: 1472


In [46]:

# 1472 zeros collected, but chunks 5-9 are PARTIAL (refined a subset of brackets, with gaps).
# The contiguous (gap-free) zero list comes from chunks 0-4, covering t in [0.5, 1038.5] = 877 zeros.
# 
# IMPORTANT: For chunks 5+, the partial refinement is the FIRST n brackets in order, which means:
# - Chunk 5 (1038.5 to 1415.8): refined first 239 of 375 brackets — these are the LOWEST 239 t-values in that range, which IS a contiguous prefix.
# - Chunk 6 (1415.8 to 1840.0): refined first 152 of 430 — contiguous prefix.
# - And so on.
# 
# So combined, the zeros up to a certain t in each chunk are correct. The COMPLETE contiguous block 
# is chunks 0-4 PLUS the prefix of chunk 5 up to whatever t it reached, etc.
# 
# Let me verify: in chunk 5 with 239 zeros, the last zero is at what t?

import mpmath
mpmath.mp.dps = 50

# Sort all_zeros by Im
all_zeros_sorted = sorted(all_zeros, key=lambda z: float(mpmath.im(z)))
print(f"First zero: Im = {mpmath.im(all_zeros_sorted[0])}")
print(f"Last zero: Im = {mpmath.im(all_zeros_sorted[-1])}")

# Examine chunk boundaries — last zero of chunk 5
# Re-read individual chunk files to know per-chunk progress
chunk_zeros = {}
for i in range(10):
 ckpt = f"/workspace/64235940-e52e-49d9-8fe9-800691f91dac/ldh_chunks/chunk_{i:02d}.pkl"
 if os.path.exists(ckpt):
 with open(ckpt, 'rb') as f:
 ck = pickle.load(f)
 chunk_zeros[i] = ck.get('zeros', [])

for i in sorted(chunk_zeros):
 zs = chunk_zeros[i]
 if zs:
 last_t = mpmath.im(zs[-1])
 chunk_end = args_list[i][1]
 print(f"Chunk {i}: {len(zs)} zeros, last t = {float(last_t):.3f} (chunk range ends at {chunk_end:.1f})")


First zero: Im = 5.0941598445710949256987955170797974750670744531091
Last zero: Im = 2838.5937593358243629316618497233180120470870709261
Chunk 0: 32 zeros, last t = 66.656 (chunk range ends at 67.3)
Chunk 1: 105 zeros, last t = 218.175 (chunk range ends at 218.7)
Chunk 2: 181 zeros, last t = 434.775 (chunk range ends at 435.8)
Chunk 3: 244 zeros, last t = 710.492 (chunk range ends at 710.6)
Chunk 4: 315 zeros, last t = 1038.094 (chunk range ends at 1038.5)
Chunk 5: 239 zeros, last t = 1280.277 (chunk range ends at 1415.8)
Chunk 6: 152 zeros, last t = 1564.244 (chunk range ends at 1840.0)
Chunk 7: 127 zeros, last t = 1963.966 (chunk range ends at 2308.9)
Chunk 8: 54 zeros, last t = 2359.018 (chunk range ends at 2820.7)
Chunk 9: 23 zeros, last t = 2838.594 (chunk range ends at 3374.0)


In [47]:

# We have GAPS between chunks 5-9. E.g., chunk 5 refined up to t=1280.3 but chunk 6 starts at t=1415.8.
# So zeros in (1280.3, 1415.8) are missing.
# 
# The CONTIGUOUS (gap-free) zero list extends only as far as chunk 4 ends (1038.5), or possibly into
# chunk 5 up to the chunk 5 endpoint of complete refinement (since each chunk processes brackets in 
# order, chunk 5's first 239 brackets cover t in [1038.5, 1280.3] continuously). Then there's a GAP 
# from 1280.3 to 1415.8 (start of chunk 6).
# 
# Therefore, the LARGEST contiguous gap-free zero list is the union of:
# - Chunks 0-4 fully complete: t in [0.5, 1038.5], 877 zeros
# - Chunk 5 prefix up to t=1280.3: 239 zeros 
# Total: 1116 zeros contiguously, covering t in [0.5, 1280.3]
# 
# That is what we can save as a verified, gap-free list of THE FIRST 1116 zeros of L_DH.
# (The 1472 total includes non-contiguous zeros, which we should NOT save as "first N zeros".)

# Build the contiguous list. Use a conservative cutoff: t-value where there's a gap.
# Sort all zeros, then walk through and stop at the first place where the cumulative count diverges 
# from N(t) expectation.

mpmath.mp.dps = 50
all_sorted = sorted(all_zeros, key=lambda z: float(mpmath.im(z)))

# Use chunks 0-4 + chunk 5 as gap-free zeros
gap_free_zeros = []
for i in range(6):
 if i in chunk_zeros:
 gap_free_zeros.extend(chunk_zeros[i])
gap_free_zeros = sorted(gap_free_zeros, key=lambda z: float(mpmath.im(z)))
print(f"Gap-free zeros (chunks 0-5): {len(gap_free_zeros)}")
print(f"First zero: t = {mpmath.im(gap_free_zeros[0])}")
print(f"Last zero: t = {mpmath.im(gap_free_zeros[-1])}")

# Verify count is consistent with Riemann-von Mangoldt prediction
import math
t_last = float(mpmath.im(gap_free_zeros[-1]))
N_predicted = (t_last/(2*math.pi))*math.log(5*t_last/(2*math.pi*math.e))
print(f"N(T={t_last:.2f}) predicted: {N_predicted:.1f}, actual: {len(gap_free_zeros)}, diff: {len(gap_free_zeros)-N_predicted:.1f}")


Gap-free zeros (chunks 0-5): 1116
First zero: t = 5.0941598445710949256987955170797974750670744531091
Last zero: t = 1280.277291430346319308931805900488051779435479727
N(T=1280.28) predicted: 1207.6, actual: 1116, diff: -91.6
